In [4]:
# 检查生成样本是否完全
import os
import shutil

os.makedirs('./undone_pdb', exist_ok=True)
pdb_done = [dir for dir in os.listdir('./PepSet_dimer_output') if len(os.listdir(os.path.join('./PepSet_dimer_output', dir))) == 15] 
print(len(pdb_done))
for file in sorted(os.listdir('PepSet_dimer')):
    if file.endswith('.pdb'):
        pdb_code = file[:-4]
        if pdb_code in pdb_done:
            print(f"{pdb_code} is done.")
        else:
            print(f"{pdb_code} is missing.")
            shutil.copy(os.path.join('./PepSet_dimer', file), os.path.join('./undone_pdb', file))




76
1a0n is done.
1cqg is done.
1czy is done.
1ddv is done.
1eg4 is done.
1f47 is done.
1f8h is done.
1j2x is done.
1jd5 is done.
1jq8 is done.
1jw6 is done.
1l2z is done.
1lb6 is done.
1mf4 is done.
1mv0 is done.
1nrl is done.
1nvq is done.
1o0p is done.
1p7v is done.
1pmx is done.
1r17 is done.
1rxz is done.
1t74 is done.
1v1t is done.
1w80 is done.
1yfn is done.
1ywi is done.
1ywo is done.
1yy6 is done.
2aij is done.
2aq9 is done.
2c3i is done.
2cch is done.
2cny is done.
2e4h is done.
2fgr is done.
2fka is done.
2hpl is done.
2ht9 is done.
2i94 is done.
2ivz is done.
2jkg is done.
2jnw is done.
2jqk is done.
2ka9 is done.
2kbr is done.
2ke1 is done.
2khh is done.
2koh is done.
2las is done.
2lsi is done.
2m0v is done.
2nnu is done.
2o9v is done.
2p1o is done.
2qos is done.
2r7g is done.
2rol is done.
2v8c is done.
2vzg is done.
2xcb is done.
2xpn is done.
2xrw is done.
2ymt is done.
3agy is done.
3bej is done.
3bqo is done.
3emh is done.
3emw is done.
3et3 is done.
3fp2 is done.
3g2

# 按照蛋白链对齐，计算蛋白和多肽分别的rmsd，合并预测数据到csv文件中

In [6]:
import os
import json
import numpy as np
import pandas as pd

OUTPUT_ROOT = "./predict/PepSet_dimer_output_esmfold2_msa_nonpairing"
REF_PDB_DIR = "./PepSet_dimer"
BACKBONE_SET = {"N", "CA", "C", "O"}
ATOM_ORDER = {"N": 0, "CA": 1, "C": 2, "O": 3}


def parse_pdb_backbone(pdb_path):
    chains = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            atom_name = line[12:16].strip()
            if atom_name not in BACKBONE_SET:
                continue
            chain_id = line[21:22]
            try:
                res_seq = int(line[22:26].strip())
            except ValueError:
                continue
            x = float(line[30:38].strip())
            y = float(line[38:46].strip())
            z = float(line[46:54].strip())
            chains.setdefault(chain_id, []).append((res_seq, atom_name, x, y, z))

    result = {}
    for cid, atoms in chains.items():
        atoms.sort(key=lambda a: (a[0], ATOM_ORDER[a[1]]))
        result[cid] = np.array([[x, y, z] for _, _, x, y, z in atoms], dtype=float)
    return result


def parse_cif_backbone(cif_path):
    col_map = {}
    chains = {}
    with open(cif_path) as f:
        lines = f.readlines()

    atom_start = -1
    for i, line in enumerate(lines):
        if line.strip() == "loop_" and i + 1 < len(lines) and lines[i + 1].strip().startswith("_atom_site."):
            atom_start = i
            break

    if atom_start == -1:
        return chains

    j = atom_start + 1
    while j < len(lines) and lines[j].strip().startswith("_atom_site."):
        col_name = lines[j].strip().split()[0]
        col_map[col_name] = len(col_map)
        j += 1

    idx_chain = col_map.get("_atom_site.auth_asym_id", 11)
    idx_atom = col_map.get("_atom_site.auth_atom_id", 12)
    idx_seq = col_map.get("_atom_site.auth_seq_id", 9)
    idx_x = col_map.get("_atom_site.Cartn_x", 14)
    idx_y = col_map.get("_atom_site.Cartn_y", 15)
    idx_z = col_map.get("_atom_site.Cartn_z", 16)
    idx_model = col_map.get("_atom_site.pdbx_PDB_model_num", 17)

    for line in lines[j:]:
        parts = line.strip().split()
        if not parts or parts[0] != "ATOM":
            continue
        if idx_model < len(parts) and parts[idx_model] != "1":
            continue
        atom_name = parts[idx_atom]
        if atom_name not in BACKBONE_SET:
            continue
        chain_id = parts[idx_chain]
        try:
            res_seq = int(parts[idx_seq])
        except (ValueError, IndexError):
            continue
        x = float(parts[idx_x])
        y = float(parts[idx_y])
        z = float(parts[idx_z])
        chains.setdefault(chain_id, []).append((res_seq, atom_name, x, y, z))

    result = {}
    for cid, atoms in chains.items():
        atoms.sort(key=lambda a: (a[0], ATOM_ORDER[a[1]]))
        result[cid] = np.array([[x, y, z] for _, _, x, y, z in atoms], dtype=float)
    return result


def kabsch_align(P, Q):
    """
    Kabsch: 求 R 和 t 使 P @ R.T + t ≈ Q.
    P, Q: (N, 3). 返回 (R, t, rmsd).
    """
    p_cent = P.mean(axis=0)
    q_cent = Q.mean(axis=0)
    Pc = P - p_cent
    Qc = Q - q_cent

    H = Pc.T @ Qc
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        R = Vt.T @ U.T

    t = q_cent - p_cent @ R.T
    P_rot = Pc @ R.T
    rmsd = float(np.sqrt(np.mean(np.sum((P_rot - Qc) ** 2, axis=1))))
    return R, t, rmsd


complexes = sorted([
    d for d in os.listdir(OUTPUT_ROOT)
    if os.path.isdir(os.path.join(OUTPUT_ROOT, d))
])

for pdb in complexes:
    ref_pdb_path = os.path.join(REF_PDB_DIR, f"{pdb}.pdb")
    complex_output = os.path.join(OUTPUT_ROOT, pdb)

    if not os.path.exists(ref_pdb_path):
        print(f"[SKIP] {pdb}: reference PDB not found")
        continue

    ref_chains = parse_pdb_backbone(ref_pdb_path)
    if "L" not in ref_chains:
        print(f"[SKIP] {pdb}: no L chain in reference")
        continue
    ref_pep = ref_chains["L"]
    ref_prot = None
    for cid, coords in ref_chains.items():
        if cid != "L":
            ref_prot = coords
            break
    if ref_prot is None:
        print(f"[SKIP] {pdb}: no non-L chain in reference")
        continue

    rows = []
    sample_dirs = sorted([
        d for d in os.listdir(complex_output)
        if d.startswith("seed-") and os.path.isdir(os.path.join(complex_output, d))
    ])

    for sample_dir in sample_dirs:
        sample_path = os.path.join(complex_output, sample_dir)
        parts = sample_dir.split("-")
        try:
            seed = int(parts[1])
            sample_id = int(parts[3])
        except (IndexError, ValueError):
            continue

        json_path = os.path.join(sample_path, "result.json")
        if not os.path.exists(json_path):
            continue
        try:
            with open(json_path) as f:
                metrics = json.load(f)
            chain_plddt_A = round(metrics["complex"]["chain_plddt_mean"]["A"], 4)
            chain_plddt_B = round(metrics["complex"]["chain_plddt_mean"]["B"], 4)
            plddt_mean = round(metrics["plddt_mean"], 4)
            ptm = round(metrics["ptm"], 4)
            iptm = round(metrics["iptm"], 4)
        except (KeyError, json.JSONDecodeError):
            continue

        cif_path = os.path.join(sample_path, "complex.cif")
        if not os.path.exists(cif_path):
            continue
        pred_chains = parse_cif_backbone(cif_path)
        if "A" not in pred_chains or "B" not in pred_chains:
            continue
        pred_prot = pred_chains["A"]
        pred_pep = pred_chains["B"]

        if len(ref_prot) != len(pred_prot):
            continue

        # 用蛋白链求 R 和 t
        R, t, protein_rmsd = kabsch_align(pred_prot, ref_prot)
        protein_rmsd = round(protein_rmsd, 4)

        # 多肽施加相同的 R 和 t
        if len(ref_pep) == 0 or len(pred_pep) == 0:
            peptide_rmsd = None
        elif len(ref_pep) != len(pred_pep):
            peptide_rmsd = None
        else:
            pred_pep_aligned = pred_pep @ R.T + t
            diffs = pred_pep_aligned - ref_pep
            peptide_rmsd = round(float(np.sqrt(np.mean(np.sum(diffs ** 2, axis=1)))), 4)

        rows.append({
            "complex": pdb,
            "seed": seed,
            "id": sample_id,
            "chain_plddt_mean_A": chain_plddt_A,
            "chain_plddt_mean_B": chain_plddt_B,
            "plddt_mean": plddt_mean,
            "ptm": ptm,
            "iptm": iptm,
            "protein_rmsd": protein_rmsd,
            "peptide_rmsd": peptide_rmsd,
        })

    if rows:
        result = pd.DataFrame(rows)
        result.to_csv(os.path.join(complex_output, "metrics_summary.csv"), index=False, na_rep="NaN")
        print(f"[OK] {pdb}: {len(rows)} samples, saved to metrics_summary.csv")
    else:
        print(f"[EMPTY] {pdb}: no valid samples")


[OK] 1a0n: 15 samples, saved to metrics_summary.csv
[OK] 1cqg: 15 samples, saved to metrics_summary.csv
[OK] 1czy: 15 samples, saved to metrics_summary.csv
[OK] 1ddv: 15 samples, saved to metrics_summary.csv
[OK] 1eg4: 15 samples, saved to metrics_summary.csv
[OK] 1f47: 15 samples, saved to metrics_summary.csv
[OK] 1f8h: 15 samples, saved to metrics_summary.csv
[OK] 1j2x: 15 samples, saved to metrics_summary.csv
[OK] 1jd5: 15 samples, saved to metrics_summary.csv
[OK] 1jq8: 15 samples, saved to metrics_summary.csv
[OK] 1jw6: 15 samples, saved to metrics_summary.csv
[OK] 1l2z: 15 samples, saved to metrics_summary.csv
[OK] 1lb6: 15 samples, saved to metrics_summary.csv
[OK] 1mf4: 15 samples, saved to metrics_summary.csv
[OK] 1mv0: 15 samples, saved to metrics_summary.csv
[OK] 1nrl: 15 samples, saved to metrics_summary.csv
[OK] 1nvq: 15 samples, saved to metrics_summary.csv
[OK] 1o0p: 15 samples, saved to metrics_summary.csv
[OK] 1p7v: 15 samples, saved to metrics_summary.csv
[OK] 1pmx: 1

In [7]:
# 合并所有的csv文件,存放在/home/junjiechen/1_work/250401-Dpepalign/Benchmark/esmfold2/PepSet_dimer_output_esmfold2下
import pandas as pd
import os
rows = []
for dir in sorted(os.listdir("./predict/PepSet_dimer_output_esmfold2_msa_nonpairing")):
    for file in sorted(os.listdir(os.path.join("./predict/PepSet_dimer_output_esmfold2_msa_nonpairing", dir))):
        if file.endswith("metrics_summary.csv"):
            df = pd.read_csv(os.path.join("./predict/PepSet_dimer_output_esmfold2_msa_nonpairing", dir, file))
            rows.append(df)
if rows:
    df_combined = pd.concat(rows, ignore_index=True)
else:
    raise ValueError("No metrics_summary.csv files found in the directory.")

df_combined.to_csv('./predict_results/esmfold2_msa_nonpairing_loop10_samples200.csv', index=False)



In [8]:
# 检查summary文件，计算每个样本的成功率。成功率的定义为每个complex存在一个多肽plddt>=70，iptm>=0.7，多肽rmsd<2.5的样本，则这个样本预测成功
import pandas as pd
df = pd.read_csv('./predict_results/esmfold2_msa_nonpairing_loop10_samples200.csv')
success_count = 0

for complex_name, group in df.groupby('complex'):
    if any(
        (group['chain_plddt_mean_B'] >= 0.7) &
        (group['iptm'] >= 0.7) &
        (group['peptide_rmsd'] < 2.5)
    ):
        success_count += 1

total_count = df['complex'].nunique()
success_rate = success_count / total_count if total_count > 0 else 0

print(f"Oracle Success count: {success_count}")
print(f"Total count: {total_count}")
print(f"Oracle Success rate: {success_rate:.2%}")

Oracle Success count: 90
Total count: 166
Oracle Success rate: 54.22%


In [9]:
# 由于esmfold2输出的cif格式无法被正确解析，其缺少一列_atom_site.occupancy，导致DockQ无法被正确计算。需要在其中添加一列_atom_site.occupancy，值为1.0。以下是一个示例脚本，可以批量处理所有的cif文件：
def add_occupancy_to_cif(cif_path):
    with open(cif_path, 'r') as f:
        lines = f.readlines()
    
    has_occupancy_header = any(l.strip().startswith('_atom_site.occupancy') for l in lines)
    new_lines = []
    for idx, line in enumerate(lines):
        if line.startswith('_atom_site.'):
            new_lines.append(line)
            # 当当前行以 "_atom_site" 开头且下一行不以 "_atom_site" 开头时，插入 occupancy header（如果还不存在）
            next_line = lines[idx + 1] if idx + 1 < len(lines) else ''
            if not has_occupancy_header and not next_line.startswith('_atom_site'):
                new_lines.append('_atom_site.occupancy\n')
                has_occupancy_header = True
        elif line.startswith('ATOM') or line.startswith('HETATM'):
            ln = line.rstrip('\n')
            new_lines.append(ln + "    1.0" + '\n')
        else:
            new_lines.append(line)
    output_file = cif_path.replace('.cif', '_with_occupancy.cif')
    with open(output_file, 'w') as f:
        f.writelines(new_lines)

for root, dirs, files in os.walk("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/esmfold2/predict/PepSet_dimer_output_esmfold2_msa_nonpairing"):
    for file in files:
        if file.endswith('.cif'):
            cif_path = os.path.join(root, file)
            add_occupancy_to_cif(cif_path)

# 汇总dockq结果和预测结果

In [10]:
# 读取DockQ/results中的csv文件以及predict_results文件，将相同的来源的csv文件按照complex，seed，sample进行合并，包含两组的所有列。在dockq的csv中complex对应为pdb
import pandas as pd
mapping = {
    "PepSet_dimer_output_esmfold2_docking_results.csv": "esmfold2_nomsa_loop10_samples200.csv",
    "PepSet_dimer_output_esmfold2_fast_docking_results.csv": "esmfold2_fast_loop10_samples200.csv",
    "PepSet_dimer_output_esmfold2_fast_loop10_samplingsteps68_docking_results.csv": "esmfold2_fast_loop10_samplingsteps68.csv",
    "PepSet_dimer_output_esmfold2_msa_nonpairing_docking_results.csv": "esmfold2_msa_nonpairing_loop10_samples200.csv",
}

for dockq_csv, predict_csv in mapping.items():
    df_dockq = pd.read_csv(f'DockQ/results/{dockq_csv}')
    df_predict = pd.read_csv(f'./predict_results/{predict_csv}')
    
    # 提取 complex, seed, sample
    # 合并数据
    df_merged = pd.merge(df_predict, df_dockq, on=['complex', 'seed', 'id'], how='inner')

    # 保存合并后的结果
    output_file = f'./summary/merged_{predict_csv}'
    df_merged.to_csv(output_file, index=False)
    print(f'Merged data saved to {output_file}')

Merged data saved to ./summary/merged_esmfold2_nomsa_loop10_samples200.csv
Merged data saved to ./summary/merged_esmfold2_fast_loop10_samples200.csv
Merged data saved to ./summary/merged_esmfold2_fast_loop10_samplingsteps68.csv
Merged data saved to ./summary/merged_esmfold2_msa_nonpairing_loop10_samples200.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# DockQ 累计堆叠柱状图
# 累计定义：Pass(>=0.23) 包含 Medium(>=0.49) 包含 High(>=0.80)
# 柱子从下到上: High → Medium-High → Pass-Medium
# oracle: 每个 complex 所有样本中取 DockQ 最高值
# model:  每个 complex 中选 iptm 最大的样本

methods = ['esmfold2_nomsa_loop10_samples200', 'esmfold2_fast_loop10_samples200', 'esmfold2_fast_loop10_samplingsteps68', 'esmfold2_msa_nonpairing_loop10_samples200']
method_labels = ['ESMFold2(no MSA)', 'ESMFold2-Fast', 'ESMFold2-Fast(Sampling Steps=68)', 'ESMFold2(MSA, Non-pairing)']

# 选择共有的 complex
all_dfs = {}
common_complexes = None
for method in methods:
    df = pd.read_csv(f'./summary/merged_{method}.csv')
    complexes = set(df['complex'].unique())
    all_dfs[method] = df
    if common_complexes is None:
        common_complexes = complexes
    else:
        common_complexes &= complexes

common_complexes = sorted(common_complexes)
print(f"Common complexes: {len(common_complexes)}")

rows = []
for method in methods:
    df = all_dfs[method]
    df = df[df['complex'].isin(common_complexes)]
    total = df['complex'].nunique()

    for mode, mode_label in [('oracle', 'Oracle'), ('model', 'Model')]:
        high = medium = low = 0
        for _, group in df.groupby('complex'):
            if mode == 'oracle':
                best_dockq = group['dockq_score'].max()
            else:
                best_dockq = group.loc[group['iptm'].idxmax(), 'dockq_score']

            if best_dockq >= 0.80:
                high += 1
            if best_dockq >= 0.49:
                medium += 1
            if best_dockq >= 0.23:
                low += 1

        rows.append({
            'method': method,
            'mode': mode_label,
            'high': high / total * 100,
            'medium': medium / total * 100,
            'acceptable': low / total * 100,
            'acceptable_err': np.sqrt((low / total) * (1 - low / total) / total) * 100,
            'total': total,
        })
        print(f"{method_labels[methods.index(method)]} {mode_label}: "
              f"High={high}/{total}={high/total*100:.1f}%, "
              f"Medium={medium}/{total}={medium/total*100:.1f}%, "
              f"Acceptable={low}/{total}={low/total*100:.1f}%")

# 绘图
df_plot = pd.DataFrame(rows)
x = np.arange(len(methods))
bar_width = 0.35

oracle_colors = {'high': '#2D5E91', 'medium': '#5C93C8', 'acceptable': '#8CB1D8'}
model_colors  = {'high': '#BF632F', 'medium': '#DD8452', 'acceptable': '#E8B88A'}
legend_labels = {
    'acceptable': 'Acceptable (DockQ ≥ 0.23)',
    'medium': 'Medium (DockQ ≥ 0.49)',
    'high': 'High (DockQ ≥ 0.80)',
}

fig, ax = plt.subplots(figsize=(9, 5.5))

for mode_offset, mode_name in enumerate(['Oracle', 'Model']):
    offset = -bar_width/2 if mode_name == 'Oracle' else bar_width/2
    sub = df_plot[df_plot['mode'] == mode_name]
    colors = oracle_colors if mode_name == 'Oracle' else model_colors

    high_vals = sub['high'].values
    med_vals = sub['medium'].values - sub['high'].values
    pass_vals = sub['acceptable'].values - sub['medium'].values

    ax.bar(x + offset, pass_vals, bar_width, bottom=high_vals + med_vals,
           label=legend_labels['acceptable'] if mode_offset == 0 else '',
           color=colors['acceptable'], edgecolor='white', linewidth=0.5)
    ax.bar(x + offset, med_vals, bar_width, bottom=high_vals,
           label=legend_labels['medium'] if mode_offset == 0 else '',
           color=colors['medium'], edgecolor='white', linewidth=0.5)
    ax.bar(x + offset, high_vals, bar_width, bottom=0,
           label=legend_labels['high'] if mode_offset == 0 else '',
           color=colors['high'], edgecolor='white', linewidth=0.5)

    total_pass = high_vals + med_vals + pass_vals

    # 仅在顶部 Acceptable 边界添加 error bar
    ax.errorbar(x + offset, total_pass, yerr=sub['acceptable_err'].values,
                fmt='none', ecolor='black', capsize=2, capthick=0.8, elinewidth=0.8)

    for i in range(len(methods)):
        ax.text(x[i] + offset, -0.5, mode_name,
                ha='center', va='top', fontsize=8, fontweight='bold',
                color='#4C72B0' if mode_name == 'Oracle' else '#DD8452')

    for i in range(len(methods)):
        ax.text(x[i] + offset, total_pass[i] + 1.5 * sub['acceptable_err'].iloc[i] + 0.8,
                f"{total_pass[i]:.1f}%", ha='center', va='bottom',
                fontsize=8, fontweight='bold')

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)

ax.set_ylabel('Percentage of Complexes (%)')
ax.set_title(f'Protein-Peptide Complex Structure Prediction Performance (n={len(common_complexes)})')
ax.set_xticks(x)
ax.set_xticklabels(method_labels, rotation=45, ha='right')
ax.set_ylim(-6.5, 115)

plt.tight_layout()
plt.savefig('./summary/dockq_quality_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Pass Rate 分组柱状图（基于三元指标）
# 评判标准：chain_plddt_mean_B >= 0.7 AND iptm >= 0.7 AND lrmsd < 2.5
# Oracle:  每个 complex 中选 peptide_rmsd 最小的样本
# Model-iptm:  每个 complex 中选 iptm 最大的样本
# Model-pep_plddt: 每个 complex 中选 chain_plddt_mean_B 最大的样本

def is_pass(row):
    return row['chain_plddt_mean_B'] >= 0.7 and row['iptm'] >= 0.7 and row['lrmsd'] < 2.5

methods = ['esmfold2_nomsa_loop10_samples200', 'esmfold2_fast_loop10_samples200', 'esmfold2_fast_loop10_samplingsteps68', 'esmfold2_msa_nonpairing_loop10_samples200']
method_labels = ['ESMFold2(no MSA)', 'ESMFold2-Fast', 'ESMFold2-Fast(Sampling Steps=68)', 'ESMFold2(MSA, Non-pairing)']

# 选择共有的 complex
all_dfs = {}
common_complexes = None
for method in methods:
    df = pd.read_csv(f'./summary/merged_{method}.csv')
    complexes = set(df['complex'].unique())
    all_dfs[method] = df
    if common_complexes is None:
        common_complexes = complexes
    else:
        common_complexes &= complexes

common_complexes = sorted(common_complexes)
print(f"Common complexes: {len(common_complexes)}")

# 三种选择模式
modes = [
    ('oracle', 'Oracle', 'peptide_rmsd', 'min'),
    ('model_iptm', 'Model-iptm', 'iptm', 'max'),
    ('model_pep_plddt', 'Model-pep_plddt', 'chain_plddt_mean_B', 'max'),
]

rows = []
for method in methods:
    df = all_dfs[method]
    df = df[df['complex'].isin(common_complexes)]
    total = df['complex'].nunique()

    for mode_key, mode_label, select_col, select_how in modes:
        pass_count = 0
        for _, group in df.groupby('complex'):
            if select_how == 'min':
                best = group.loc[group[select_col].idxmin()]
            else:
                best = group.loc[group[select_col].idxmax()]

            if is_pass(best):
                pass_count += 1

        pass_rate = pass_count / total * 100
        se = np.sqrt((pass_count / total) * (1 - pass_count / total) / total) * 100

        rows.append({
            'method': method,
            'mode': mode_label,
            'pass_rate': pass_rate,
            'pass_err': se,
            'total': total,
        })
        print(f"{method_labels[methods.index(method)]} {mode_label}: "
              f"Pass={pass_count}/{total}={pass_rate:.1f}%")

# 绘图
df_plot = pd.DataFrame(rows)
x = np.arange(len(methods))
bar_width = 0.25

colors = {'Oracle': '#4C72B0', 'Model-iptm': '#DD8452', 'Model-pep_plddt': '#55A868'}
mode_labels = ['Oracle', 'Model-iptm', 'Model-pep_plddt']

fig, ax = plt.subplots(figsize=(10, 5.5))

for i, mode_label in enumerate(mode_labels):
    offset = (i - 1) * bar_width
    sub = df_plot[df_plot['mode'] == mode_label]
    vals = sub['pass_rate'].values
    errs = sub['pass_err'].values

    ax.bar(x + offset, vals, bar_width,
           label=mode_label, color=colors[mode_label],
           edgecolor='white', linewidth=0.5)

    ax.errorbar(x + offset, vals, yerr=errs,
                fmt='none', ecolor='black', capsize=2, capthick=0.8, elinewidth=0.8)

    for j in range(len(methods)):
        err_top = max(errs[j], 1.0)
        ax.text(x[j] + offset, vals[j] + err_top + 1.0,
                f"{vals[j]:.1f}%", ha='center', va='bottom',
                fontsize=7.5, fontweight='bold')

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)

ax.set_ylabel('Pass Rate (%)')
ax.set_title(f'Protein-Peptide Complex Structure Prediction Performance (n={len(common_complexes)})')
ax.set_xticks(x)
ax.set_xticklabels(method_labels, rotation=45, ha='right')
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('./summary/pass_rate_comparison.png', dpi=300, bbox_inches='tight')
plt.show()